<a id="top"></a>

# Skills demo: a hand-rolled just-in-time loader

```yaml
title: "Skills demo: a hand-rolled just-in-time loader"
keywords: skill, jit loading, just-in-time, progressive disclosure, skill catalog, load_skill, description, trigger, token cost, always-on context, fake model, offline
estimated duration: 20
```

> **Lesson:** L22. **Roadmap:** [demos_or_activities.md](../../../../docs/origin/lesson_roadmaps/L22/demos_or_activities.md).
> A teacher-led **live build**: we hand-roll the just-in-time skill loader the lecture
> ([L2202_lecture.md](L2202_lecture.md)) described, on the same model->act->model shape you
> built in L10 and L11. It runs fully **offline** on the scripted `FakeModel` (no API key),
> so the *mechanism* -- not the model's mood -- is what you watch.
> **Anchor model: Claude Sonnet 4.6** in production; here the deterministic `FakeModel`
> stands in so every run is identical.

## Contents

- [1. Setup](#1-setup)
- [2. The catalog](#2-the-catalog)
- [3. A minimal JIT loader](#3-a-minimal-jit-loader)
- [4. Watch it load just in time](#4-watch-it-load-just-in-time)
- [5. The money slide](#5-the-money-slide)
- [6. The costs you do pay](#6-the-costs-you-do-pay)

## 1. Setup

Everything here is **given**: the deterministic `FakeModel` from `common` (our offline stand-in for the Anthropic client), a rough token counter, and a shelf of five `Skill`s. A skill is just a `name`, a `description` that says **when** it applies, and a `body` of instructions. Run this cell top-to-bottom; no edits, no API key.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Annotated

from langchain_core.messages import HumanMessage

from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)


def rough_tokens(text: str) -> int:
    """Back-of-envelope token count: ~4 characters per token.

    Not exact -- real tokenization is the model's business (L01) -- but good
    enough to *compare* always-on vs. just-in-time context sizes.
    """
    return len(text) // 4


@dataclass(frozen=True)
class Skill:
    """A skill: a name, a description that says WHEN it applies, and a body of
    instructions that is loaded only once the skill is selected."""

    name: str
    description: str
    body: str


# A tiny shelf of five skills. Only `name` + `description` stay in context by
# default; each `body` sits here on "disk" until the model asks to load it.
SKILLS: dict[str, Skill] = {
    "refund-policy": Skill(
        name="refund-policy",
        description=(
            "Apply when the user asks to process a refund, return, or money-back "
            "request. Covers the return window, restocking fees, and how to phrase the reply."
        ),
        body=(
            "# Refund policy\n"
            "1. Confirm the order is within the 30-day return window.\n"
            "2. Refunds go to the original payment method only.\n"
            "3. Restocking fee: 10% on opened items, 0% on unopened.\n"
            "4. Reply politely: state the refund amount and the 5-7 day timeline.\n"
        ),
    ),
    "warranty-claim": Skill(
        name="warranty-claim",
        description=(
            "Apply when the user reports a defective product still under warranty and "
            "wants a repair or replacement (not a refund)."
        ),
        body=(
            "# Warranty claim\n"
            "1. Confirm the purchase is within 12 months.\n"
            "2. Ask for a photo or description of the defect.\n"
            "3. Offer repair first, replacement if unrepairable.\n"
        ),
    ),
    "subscription-cancel": Skill(
        name="subscription-cancel",
        description=(
            "Apply when the user asks to cancel or pause a recurring subscription or membership."
        ),
        body=(
            "# Subscription cancellation\n"
            "1. Confirm the plan and the next billing date.\n"
            "2. Offer a pause before a full cancel.\n"
            "3. Confirm cancellation and the final access date.\n"
        ),
    ),
    "shipping-eta": Skill(
        name="shipping-eta",
        description=(
            "Apply when the user asks where their order is or when it will arrive "
            "(delivery estimates and tracking)."
        ),
        body=(
            "# Shipping ETA\n"
            "1. Standard shipping is 5-7 business days; express is 2.\n"
            "2. Look up the tracking number and quote the carrier's estimate.\n"
        ),
    ),
    "bulk-quote": Skill(
        name="bulk-quote",
        description=(
            "Apply when the user asks for volume pricing or a quote for a large or wholesale order."
        ),
        body=(
            "# Bulk order quote\n"
            "1. Orders over 50 units get 15% off; over 200 get 25%.\n"
            "2. Collect the SKU and quantity, then compute the discounted total.\n"
        ),
    ),
}

[↑ Back to top](#top)

## 2. The catalog

The **catalog** is the always-on view: every skill's `name` + `description`, and nothing else. This one string is *all* the model sees by default -- the bodies stay out of context until a skill is loaded. Five skills cost only five one-line descriptions.

In [2]:
def skill_catalog(skills: dict[str, Skill]) -> str:
    """The always-on view: every skill's name + description, nothing more.

    This one string is ALL the model sees by default -- the bodies stay out of
    context until a skill is loaded.
    """
    lines = [f"- {s.name}: {s.description}" for s in skills.values()]
    return "\n".join(lines)


catalog = skill_catalog(SKILLS)
print(catalog)
print()
print(f"always in context: ~{rough_tokens(catalog)} tokens for all {len(SKILLS)} skills")

- refund-policy: Apply when the user asks to process a refund, return, or money-back request. Covers the return window, restocking fees, and how to phrase the reply.
- warranty-claim: Apply when the user reports a defective product still under warranty and wants a repair or replacement (not a refund).
- subscription-cancel: Apply when the user asks to cancel or pause a recurring subscription or membership.
- shipping-eta: Apply when the user asks where their order is or when it will arrive (delivery estimates and tracking).
- bulk-quote: Apply when the user asks for volume pricing or a quote for a large or wholesale order.

always in context: ~157 tokens for all 5 skills


[↑ Back to top](#top)

## 3. A minimal JIT loader

Now the loader. It is the **same model->act->model loop** you built in L10 and drew as a graph in L11 -- except the only *action* the model can take is **loading a skill body into context**. The model is handed the catalog (in the system prompt) plus one tool, `load_skill`. We record the context size **before each turn**, so the jump when a body loads is visible in numbers.

In [3]:
@dataclass
class JitResult:
    """What one loader run reports: the final answer, which skills were loaded
    (in order), and the context size in tokens BEFORE each model turn -- so the
    jump when a body loads is visible."""

    final_text: str
    loaded: list[str]
    context_tokens: list[int]


def jit_run(
    model: FakeModel,
    skills: dict[str, Skill],
    user_msg: str,
    *,
    max_steps: int = 4,
) -> JitResult:
    """A minimal model -> act -> model loop -- the same shape as L10/L11 -- whose
    only "action" is loading a skill body into context just in time.
    """

    def load_skill(
        name: Annotated[str, "the exact skill name from the catalog to read into context"],
    ) -> str:
        """Progressive disclosure, step 2: read a skill's full body into context.

        Returns the body for `name`, or a short error string if there is no such skill.
        """
        skill = skills.get(name)
        if skill is None:
            return f"no such skill: {name!r}"
        return skill.body

    # The one tool the model gets: not the skills themselves, just the ability to
    # *read* one on demand. `bind_tools` infers the tool definition (name, docstring,
    # the typed `name` argument) straight from the function -- no hand-written JSON
    # schema. It closes over `skills`, so a call reads from THIS shelf.
    model_with_tools = model.bind_tools([load_skill])

    system = (
        "You are a support agent. Skills are listed below as `name: description`. "
        "When a skill's description matches the task, call load_skill(name) to read "
        "its full instructions, then follow them.\n\n"
        f"Available skills:\n{skill_catalog(skills)}"
    )
    transcript: list[str] = [f"USER: {user_msg}"]
    loaded: list[str] = []
    context_tokens: list[int] = []
    final_text = ""

    for _ in range(max_steps):
        # Everything currently in the window: the system prompt (catalog!) plus
        # the running transcript. Measure it BEFORE the call, each turn.
        context = system + "\n" + "\n".join(transcript)
        context_tokens.append(rough_tokens(context))

        resp = model_with_tools.invoke([HumanMessage(context)])

        if not resp.tool_calls:
            # No load requested -> the model is answering (an AIMessage with text
            # and no tool_calls). Done.
            final_text = str(resp.content)
            break

        for call in resp.tool_calls:
            if call["name"] == "load_skill":
                name = str(call["args"]["name"])
                body = load_skill(name)
                loaded.append(name)
                # The body enters the window HERE -- progressive disclosure step 2.
                transcript.append(f"ASSISTANT: (loading skill {name!r})")
                transcript.append(f"SKILL[{name}]:\n{body}")

    return JitResult(final_text=final_text, loaded=loaded, context_tokens=context_tokens)

[↑ Back to top](#top)

## 4. Watch it load just in time

Pose a refund task. The scripted model asks to load `refund-policy` (turn 1), then answers using it (turn 2). Watch `context_tokens`: turn 1 is **catalog only**; turn 2 jumps because the refund **body** is now in the window -- loaded *just in time*, only because the task matched its description.

In [4]:
# Script the model: turn 1 asks to load the refund skill; turn 2 answers using it.
refund_model = FakeModel(
    [
        tool_reply(tool_call("s1", "load_skill", {"name": "refund-policy"})),
        text_reply(
            "You're within the 30-day window. Your opened blender has a 10% "
            "restocking fee, so you'll be refunded $54 to your original card in 5-7 days."
        ),
    ]
)

task = "I want to return my opened blender and get my money back."
result = jit_run(refund_model, SKILLS, task)
print("skills loaded:", result.loaded)
print("context tokens per turn:", result.context_tokens)
print("  turn 1 (catalog only, before load):", result.context_tokens[0])
print("  turn 2 (refund body now loaded):   ", result.context_tokens[1])
print()
print("answer:", result.final_text)

skills loaded: ['refund-policy']
context tokens per turn: [225, 304]
  turn 1 (catalog only, before load): 225
  turn 2 (refund body now loaded):    304

answer: You're within the 30-day window. Your opened blender has a 10% restocking fee, so you'll be refunded $54 to your original card in 5-7 days.


And the flip side -- a task **no** skill matches. The model answers directly, nothing loads, and the context never grows past the catalog. This is the whole point: capability you *don't* use costs you nothing but its one-line description.

In [5]:
# A task no skill matches: the model answers directly, nothing loads.
lean_model = FakeModel([text_reply("2 + 2 is 4.")])
lean = jit_run(lean_model, SKILLS, "What is 2 + 2?")
print("skills loaded:", lean.loaded)  # []
print("context tokens:", lean.context_tokens, "-- catalog only, no body ever loaded")
print("answer:", lean.final_text)

skills loaded: []
context tokens: [214] -- catalog only, no body ever loaded
answer: 2 + 2 is 4.


[↑ Back to top](#top)

## 5. The money slide

Here is L07's *"tools cost tokens twice over"* problem, solved. Suppose we had instead crammed all five skill **bodies** into the system prompt (always-on). Compare that against JIT-loading only the one skill a task needed -- and against an idle call that loads nothing.

In [6]:
# All five bodies always-on vs. JIT-loading only what a task needs.
all_bodies = sum(rough_tokens(s.body) for s in SKILLS.values())
catalog_tokens = rough_tokens(skill_catalog(SKILLS))
one_body = rough_tokens(SKILLS["refund-policy"].body)

always_on = catalog_tokens + all_bodies  # every body, every call
jit = catalog_tokens + one_body  # catalog always + the one loaded

print(f"all 5 skills always-on (bodies in prompt): ~{always_on} tokens, EVERY call")
print(f"5 skills JIT (catalog + the one used):     ~{jit} tokens on the call that used one")
print(f"idle call (nothing loaded):                ~{catalog_tokens} tokens")
print()
print(f"kept out of the window on an idle call: ~{always_on - catalog_tokens} tokens")

all 5 skills always-on (bodies in prompt): ~369 tokens, EVERY call
5 skills JIT (catalog + the one used):     ~219 tokens on the call that used one
idle call (nothing loaded):                ~157 tokens

kept out of the window on an idle call: ~212 tokens


[↑ Back to top](#top)

## 6. The costs you do pay

Skills are cheap, not free. You pay an **extra model turn** to load one, and -- worse -- you risk the model **failing to load** a skill it needed. Here the model skips the load and answers generically; the refund body never enters context, so the reply misses the 10% restocking rule. A vague description (or a distractible model) causes exactly this **silent miss** -- the failure mode from the lecture.

In [7]:
# The model skips the load and answers generically -- the silent miss.
missed_model = FakeModel(
    [text_reply("I can help with your refund -- please contact our support team.")]
)
missed = jit_run(missed_model, SKILLS, task)
print("skills loaded:", missed.loaded)  # [] -- the skill silently never fired
print("answer:", missed.final_text)
print()
print("The refund-policy body never loaded, so the reply misses the 10% restocking rule.")
print("A vague description (or a distractible model) causes exactly this silent miss.")

skills loaded: []
answer: I can help with your refund -- please contact our support team.

The refund-policy body never loaded, so the reply misses the 10% restocking rule.
A vague description (or a distractible model) causes exactly this silent miss.


**What you just built is the real thing, in miniature.** A registry of `name + description`; a `load_skill` step that pulls a body into context only when selected; a token readout that proves the savings. The next lecture ([L2205_lecture.md](L2205_lecture.md)) decides **where a capability belongs** and shows this exact shape as **Anthropic Agent Skills** -- the format *this curriculum* is authored with.

[↑ Back to top](#top)